---
title: "GSB5544_lab4"
format:
  html:
    theme: flatly
    embed-resources: true
execute:
  echo: true
---

Part 1. Data from unstructured websites
Option A
This website contains many weekly meal plans. Choose one that seems delicious to you. Scrape the weekly meal plan into to create a Pandas DataFrame with the following columns:

Day of the Week
Name of Recipe
Link to Recipe
Price of Recipe
If you get blocked, you might need to pass headers or use Session() like below. You should only run commands like this once. If you run it too many times in a row you might get blocked, and you might need to wait to try again.

I think that meal plan 215 looks yummy.

In [ ]:
import pandas as pd
from curl_cffi import requests
from bs4 import BeautifulSoup

url = "https://tastesbetterfromscratch.com/meal-plan-215/"

response = requests.get(url, impersonate="chrome")

print(response.status_code)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

200


In [4]:
# find the tables on this page
tables = soup.find_all("table")
len(tables)


0

In [7]:
# scrape the weekly meal plans
# create pd data frame with day of the week, name of recipe, link to recipe, and price of recipe
# does not use <tr
# we use <p> for this website because each meal is contained in a paragraph
import time 
import re
# Look through all paragraph elements
rows = []
for meal in soup.find_all("p"):

    # Find the weekday and recipe link
    day_tag = meal.find("strong")
    recipe_tag = meal.find("a", href=True)

    # Ignore paragraphs that are not meal entries
    if not day_tag or not recipe_tag:
        continue

    # Get all text from the paragraph
    meal_text = meal.get_text(" ", strip=True)

    # Find a price such as $5, $14, or $25
    price_match = re.search(r"\$([\d,.]+)", meal_text)

    if not price_match:
        continue

    # Extract the information
    day = day_tag.get_text(strip=True).replace(":", "")
    recipe_name = recipe_tag.get_text(strip=True)
    recipe_link = recipe_tag["href"]
    price = float(price_match.group(1).replace(",", ""))

    # Add one row to the list
    rows.append({
        "Day of the Week": day,
        "Name of Recipe": recipe_name,
        "Link to Recipe": recipe_link,
        "Price of Recipe": price
    })

time.sleep(0.5
)
# Create the DataFrame
df = pd.DataFrame(rows)

print(df.to_string(index=False))

        

Day of the Week         Name of Recipe                                                                  Link to Recipe  Price of Recipe
         Monday      Carne Asada Tacos                          https://tastesbetterfromscratch.com/carne-asada-tacos/             24.0
        Tuesday  General Tso's Chicken https://tastesbetterfromscratch.com/general-tsos-chicken-slow-cooker-stove-top/             15.0
      Wednesday Creamy Tortellini Soup  https://tastesbetterfromscratch.com/creamy-tomato-and-spinach-tortellini-soup/             13.0
       Thursday         Ham Fried Rice                             https://tastesbetterfromscratch.com/ham-fried-rice/              9.0
         Friday  Honey Mustard Chicken                      https://tastesbetterfromscratch.com/honey-mustard-chicken/             12.0


## Part 2. Data from an API
Using the Tasty API from the practice activity, search for recipes that resemble the “Monday” recipe in your meal plan. When seaching for recipes on Tasty, try some fuzzy matching; it doesn’t have to be an exact match. For example, if the Monday recipe is “biscuit chicken pot pie” you might search for “pot pie” or “biscuit chicken”. You may need to use some text editing tricks to convert the recipe name into a searchable string.

Your free Tasty API account only allows 500 queries per month. You should not need more than this, but if you do end up with a recipe that has a large number of matches, you only need to scrape the first 100 of them. It’s also fine if there are fewer than 100 matches.

In [8]:
import requests
# tasty api from the PA
domain = "https://tasty.p.rapidapi.com"
endpoint = "recipes/list"
url = f"{domain}/{endpoint}"

#querystring = {"from":"0","size":"20","q":"daikon"}



meal_plan = {
    "Monday": "Carne Asada Tacos",
    "Tuesday": "General Tso's Chicken",
    "Wednesday": "Creamy Tortellini Soup",
    "Thursday": "Ham Fried Rice",
    "Friday": "Honey Mustard Chicken",
}

headers = {
    "X-RapidAPI-Key": "b9ce2780f7msh755640312df97d0p1ef93ejsn257eb1147ccd",
    "X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

all_results = {}

for day, recipe_name in meal_plan.items():
    query = recipe_name.lower().strip()

    response = requests.get(
        "https://tasty.p.rapidapi.com/recipes/list",
        headers=headers,
        params={
            "q": query,
            "from": 0,
            "size": 100
        }
    )

    print(day, response.status_code)

    if response.ok:
        data = response.json()
        all_results[day] = data.get("results", [])
    else:
        all_results[day] = []

print({
    day: len(results)
    for day, results in all_results.items()
})


Monday 200
Tuesday 200
Wednesday 200
Thursday 200
Friday 200
{'Monday': 2, 'Tuesday': 2, 'Wednesday': 1, 'Thursday': 4, 'Friday': 27}
